In [1]:
import pandas as pd

df1 = pd.read_csv("../data/raw/online_retail_2009_2010.csv")
df2 = pd.read_csv("../data/raw/online_retail_2010_2011.csv")

df = pd.concat([df1, df2], ignore_index=True)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.shape)
df.head()

(1067371, 8)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [12]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()


Rows: 1067371
Columns: 8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   invoice      1067371 non-null  object 
 1   stockcode    1067371 non-null  object 
 2   description  1062989 non-null  object 
 3   quantity     1067371 non-null  int64  
 4   invoicedate  1067371 non-null  object 
 5   price        1067371 non-null  float64
 6   customer_id  824364 non-null   float64
 7   country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [13]:
missing = df.isnull().sum()

missing_percent = (missing / len(df)) * 100

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

missing_table

,missing_count,missing_percent
invoice,0,0.000000
stockcode,0,0.000000
description,4382,0.410541
quantity,0,0.000000
invoicedate,0,0.000000
price,0,0.000000
customer_id,243007,22.766873
country,0,0.000000


In [14]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate percentage:", (duplicate_rows / len(df)) * 100)

Duplicate rows: 34335
Duplicate percentage: 3.2167821685243467


In [15]:
negative_quantity = (df["quantity"] < 0).sum()
zero_or_negative_price = (df["price"] <= 0).sum()

print("Negative quantity rows:", negative_quantity)
print("Zero or negative price rows:", zero_or_negative_price)

Negative quantity rows: 22950
Zero or negative price rows: 6207


In [16]:
df["invoice"] = df["invoice"].astype(str)

cancelled_rows = df["invoice"].str.startswith("C").sum()

print("Cancelled invoice rows:", cancelled_rows)

Cancelled invoice rows: 19494


In [17]:
df_clean = df.copy()

df_clean = df_clean.drop_duplicates()

df_clean = df_clean[df_clean["quantity"] > 0]
df_clean = df_clean[df_clean["price"] > 0]
df_clean = df_clean[~df_clean["invoice"].str.startswith("C")]
df_clean = df_clean.dropna(subset=["description"])

print("Original rows:", df.shape[0])
print("Clean rows:", df_clean.shape[0])
print("Rows removed:", df.shape[0] - df_clean.shape[0])

df_clean.head()

Original rows: 1067371
Clean rows: 1007913
Rows removed: 59458


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [18]:
df_clean["revenue"] = df_clean["quantity"] * df_clean["price"]

df_clean["invoicedate"] = pd.to_datetime(df_clean["invoicedate"])

df_clean["year"] = df_clean["invoicedate"].dt.year
df_clean["month"] = df_clean["invoicedate"].dt.month
df_clean["month_year"] = df_clean["invoicedate"].dt.to_period("M").astype(str)

df_clean[["invoice", "quantity", "price", "revenue", "invoicedate", "year", "month", "month_year"]].head()

,invoice,quantity,price,revenue,invoicedate,year,month,month_year
0,489434,12,6.95,83.4,2009-12-01 07:45:00,2009,12,2009-12
1,489434,12,6.75,81.0,2009-12-01 07:45:00,2009,12,2009-12
2,489434,12,6.75,81.0,2009-12-01 07:45:00,2009,12,2009-12
3,489434,48,2.10,100.8,2009-12-01 07:45:00,2009,12,2009-12
4,489434,24,1.25,30.0,2009-12-01 07:45:00,2009,12,2009-12


In [19]:
total_revenue = df_clean["revenue"].sum()
total_orders = df_clean["invoice"].nunique()
total_products = df_clean["stockcode"].nunique()
total_customers = df_clean["customer_id"].nunique()
total_countries = df_clean["country"].nunique()

print("Total Revenue:", round(total_revenue, 2))
print("Total Orders:", total_orders)
print("Total Products:", total_products)
print("Total Customers:", total_customers)
print("Total Countries:", total_countries)

Total Revenue: 20476260.45
Total Orders: 40077
Total Products: 4917
Total Customers: 5878
Total Countries: 43


In [20]:
print("Start date:", df_clean["invoicedate"].min())
print("End date:", df_clean["invoicedate"].max())

Start date: 2009-12-01 07:45:00
End date: 2011-12-09 12:50:00


In [26]:
df_clean.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

print("Cleaned file saved successfully")
print(df_clean.shape)

Cleaned file saved successfully
(1007913, 12)
